# S&P 500 structured sustainability score

This notebook combines Bloomberg ESG fields with an annual regulatory panel to calculate company-level sustainability and transition scores for 500 S&P 500 companies.

The workflow has four stages: load and validate the data, build comparable features, calculate scores, and export results.

## 1. Setup

Define imports, file locations, scoring settings, materiality, and weights.

In [1]:
from pathlib import Path
from datetime import date
import hashlib
import json
import math
import os
import re
import sys
import warnings

import numpy as np
import pandas as pd

# Optional package path used by managed notebook environments.
package_fallback = os.environ.get("NOTEBOOK_PYTHON_PACKAGES")
if package_fallback:
    sys.path.append(package_fallback)

import openpyxl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", context="notebook")

### File locations

In [2]:
# Look for inputs in an environment variable, data/, the project root, or Colab.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "notebooks").exists() and (PROJECT_ROOT.parent / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent


def resolve_input(env_name, filename):
    """Return the first available path for one required input file."""
    candidates = [
        Path(os.environ[env_name]).expanduser() if os.environ.get(env_name) else None,
        PROJECT_ROOT / "data" / filename,
        PROJECT_ROOT / filename,
        Path("/content") / filename,
    ]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate
    raise FileNotFoundError(f"Set {env_name} or place {filename} in data/, the project root, or /content")


BLOOMBERG_PATH = resolve_input("BLOOMBERG_ESG_XLSX", "ESGData.xlsx")
PANEL_PATH = resolve_input("SP500_REGULATORY_PANEL_CSV", "sp500_esg_annual_features_2012_2024.csv")

OUTPUT_DIR = PROJECT_ROOT / "output" / "structured_score"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Scoring settings

In [3]:
QUANTITATIVE_ANCHOR_YEAR = 2024
GENERAL_FEATURES_TIME_BASIS = "latest/current workbook snapshot"
SCORING_AS_OF_DATE = date.today().isoformat()
TREND_YEARS = list(range(2019, 2025))
REGULATORY_YEARS = list(range(2020, 2025))
REGULATORY_RECENCY_WEIGHTS = {2020: 1, 2021: 2, 2022: 3, 2023: 4, 2024: 5}

MIN_PEER_COUNT = 15
MIN_REGULATORY_PEER_COUNT = 8
SHRINKAGE_EVIDENCE_THRESHOLD = 0.70
NEUTRAL_PRIOR = 50.0
ENVIRONMENT_SECTOR_MIX_LAMBDA = 0.50
REGULATORY_PENALTY_CAP = 15.0
CONFIDENCE_THRESHOLDS = {"High": 0.80, "Medium": 0.50}
SPARSE_COVERAGE_THRESHOLD = 0.10
HIGH_REDUNDANCY_SPEARMAN = 0.90

SECTORS = [
    "Communication", "Consumer Discretionary", "Consumer Staples", "Energy",
    "Financials", "Health Care", "Industrials", "Information Technology",
    "Materials", "Real Estate", "Utilities",
]
MATERIALITY_VALUES = {"High": 1.0, "Medium": 0.5, "Low": 0.25, "N/A": 0.0}

def levels(default="Medium", overrides=None):
    result = {sector: default for sector in SECTORS}
    result.update(overrides or {})
    return result

### Materiality

In [4]:
FEATURE_MATERIALITY_LEVELS = {
    "scope12_reported_amount_2024": levels("Medium", {
        "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High",
        "Financials": "Low", "Information Technology": "Low", "Communication": "Low",
    }),
    "scope12_employee_intensity_trend": levels("Medium", {
        "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High",
        "Financials": "Low", "Information Technology": "Low", "Communication": "Low",
    }),
    "sbti_status": levels("Medium", {
        "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High",
    }),
    "climate_governance_support": levels("Medium", {
        "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High",
    }),
    "diversity": levels("Medium", {
        "Communication": "High", "Consumer Discretionary": "High", "Consumer Staples": "High", "Financials": "High",
    }),
    "employee_safety": levels("Low", {
        "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High",
        "Consumer Discretionary": "Medium", "Consumer Staples": "Medium", "Health Care": "Medium",
    }),
    "social_policy": levels("High"),
    "employee_stability": levels("Medium"),
    "board_independence": levels("High"),
    "ceo_separation": levels("High"),
    "board_attendance": levels("High"),
    "women_executives": levels("High"),
    "sustainability_committee": levels("Medium"),
    "scope12_absolute_trend": levels("Medium", {
        "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High",
        "Financials": "Low", "Information Technology": "Low", "Communication": "Low",
    }),
}
FEATURE_MATERIALITY = {
    feature: {sector: MATERIALITY_VALUES[level] for sector, level in mapping.items()}
    for feature, mapping in FEATURE_MATERIALITY_LEVELS.items()
}

In [5]:
REGULATORY_MATERIALITY_LEVELS = {
    "tri": levels("N/A", {
        "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High",
        "Consumer Staples": "High", "Consumer Discretionary": "Medium", "Health Care": "Medium",
        "Information Technology": "Medium",
    }),
    "cfpb": levels("N/A", {"Financials": "High", "Consumer Discretionary": "Medium", "Communication": "Low"}),
    "cpsc": levels("N/A", {
        "Consumer Discretionary": "High", "Consumer Staples": "High", "Industrials": "Medium",
        "Information Technology": "Medium", "Health Care": "Low",
    }),
    "openfda": levels("N/A", {"Health Care": "High", "Consumer Staples": "Medium", "Industrials": "Low"}),
}
REGULATORY_MATERIALITY = {
    source: {sector: MATERIALITY_VALUES[level] for sector, level in mapping.items()}
    for source, mapping in REGULATORY_MATERIALITY_LEVELS.items()
}

### Weights

In [6]:
ORIGINAL_FEATURE_WEIGHTS = {
    "Environmental": {
        "scope12_revenue_intensity": 0.40, "scope3_revenue_intensity": 0.20,
        "energy_revenue_intensity": 0.20, "resource_revenue_intensity": 0.10,
        "renewable_energy_ratio": 0.10,
    },
    "Transition": {
        "scope12_employee_intensity_trend": 0.50, "sbti_status": 0.35,
        "climate_governance_support": 0.15,
    },
    "Social": {
        "diversity": 0.35, "employee_safety": 0.30, "social_policy": 0.20,
        "employee_stability": 0.15,
    },
    "Governance": {
        "board_independence": 0.35, "ceo_separation": 0.20, "board_attendance": 0.15,
        "women_executives": 0.15, "sustainability_committee": 0.15,
    },
}
FINAL_FEATURE_WEIGHTS = {
    # Use the verified same-field footprint feature for the environmental pillar.
    "Environmental": {"scope12_reported_amount_2024": 1.00},
    "Transition": ORIGINAL_FEATURE_WEIGHTS["Transition"].copy(),
    # Give safety counts a lower weight and reallocate the balance to diversity and policies.
    "Social": {"diversity": 0.40, "employee_safety": 0.20, "social_policy": 0.25, "employee_stability": 0.15},
    "Governance": ORIGINAL_FEATURE_WEIGHTS["Governance"].copy(),
}
ORIGINAL_PILLAR_WEIGHTS = {"Environmental": 0.45, "Transition": 0.15, "Social": 0.20, "Governance": 0.20}
FINAL_PILLAR_WEIGHTS = ORIGINAL_PILLAR_WEIGHTS.copy()

ORIGINAL_NET_ZERO_WEIGHTS = {
    "scope12_current_intensity": 0.25, "scope12_employee_intensity_trend": 0.25,
    "scope12_absolute_trend": 0.25, "sbti_status": 0.15, "climate_governance_support": 0.10,
}
# Build the net-zero score from comparable trend and commitment signals.
FINAL_NET_ZERO_WEIGHTS = {
    "scope12_employee_intensity_trend": 1/3, "scope12_absolute_trend": 1/3,
    "sbti_status": 0.20, "climate_governance_support": 2/15,
}
COMMITMENT_WEIGHTS = {"sbti_status": 0.65, "climate_governance_support": 0.35}
REALIZED_TRANSITION_WEIGHTS = {
    "scope12_reported_amount_2024": 0.35, "scope12_employee_intensity_trend": 0.35,
    "tri_pollution_outcome": 0.30,
}
REGULATORY_SOURCE_WEIGHTS = {"tri": 0.40, "cfpb": 0.20, "cpsc": 0.20, "openfda": 0.20}

In [7]:
ASSESSMENT_DESCRIPTION = (
    "Current structured sustainability assessment using a 2024 quantitative baseline "
    "and latest available governance and policy information."
)

print("Inputs:", BLOOMBERG_PATH.name, PANEL_PATH.name)
print("Output:", OUTPUT_DIR.relative_to(PROJECT_ROOT))
print("Scoring as of:", SCORING_AS_OF_DATE)

Inputs: ESGData.xlsx sp500_esg_annual_features_2012_2024.csv
Output: output/structured_score
Scoring as of: 2026-09-13


## 2. Load and validate data

Read the source files, standardize missing values, and confirm the expected structure.

In [8]:
BLOOMBERG_ERROR_RE = re.compile(r"^#(?:N/A|VALUE!|REF!|DIV/0!|NAME\?|NUM!|NULL!|SPILL!|CALC!)", re.I)


def clean_bloomberg_value(value):
    """Convert Bloomberg error strings to missing values and preserve real zeros."""
    if isinstance(value, str) and BLOOMBERG_ERROR_RE.match(value.strip()):
        return np.nan
    return value


def read_bloomberg_sheet(worksheet):
    """Read populated Bloomberg rows and return a small cleaning summary."""
    header = [
        str(value).strip() if value is not None else ""
        for value in next(worksheet.iter_rows(min_row=1, max_row=1, values_only=True))
    ]
    rows = []
    skipped_rows = 0
    error_cells = 0
    numeric_zeroes = 0

    for raw in worksheet.iter_rows(min_row=2, values_only=True):
        # Ignore blank rows and formula artifacts below the populated table.
        if not any(value is not None and str(value).strip() for value in raw):
            skipped_rows += 1
            continue

        cleaned = []
        for value in raw:
            error_cells += int(isinstance(value, str) and bool(BLOOMBERG_ERROR_RE.match(value.strip())))
            numeric_zeroes += int(isinstance(value, (int, float)) and not isinstance(value, bool) and value == 0)
            cleaned.append(clean_bloomberg_value(value))

        # Valid data rows start with a Bloomberg security identifier.
        security_id = cleaned[0]
        if not isinstance(security_id, str) or not security_id.strip().endswith(" Equity"):
            skipped_rows += 1
            continue
        rows.append(cleaned)

    frame = pd.DataFrame(rows, columns=header)
    diagnostics = {
        "populated_security_rows": len(frame),
        "skipped_empty_or_artifact_rows": skipped_rows,
        "converted_error_cells": error_cells,
        "preserved_numeric_zero_cells": numeric_zeroes,
    }
    return frame, diagnostics

In [9]:
def normalize_ticker(value):
    """Convert Bloomberg ticker formats to the panel's ticker format."""
    text = str(value).upper().strip()
    text = re.sub(r"\s+[A-Z]{2}\s+EQUITY$", "", text)
    text = re.sub(r"\s+EQUITY$", "", text)
    text = text.replace("/", " ").replace(".", " ")
    return re.sub(r"\s+", " ", text).strip()


def values_equal(left, right):
    """Compare two values while treating two missing values as equal."""
    if pd.isna(left) and pd.isna(right):
        return True
    if pd.isna(left) or pd.isna(right):
        return False
    if isinstance(left, (int, float, np.number)) and isinstance(right, (int, float, np.number)):
        return bool(np.isclose(float(left), float(right), rtol=1e-10, atol=1e-12))
    return str(left).strip() == str(right).strip()


def sha256(path):
    """Create a reproducible fingerprint for an input file."""
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

### Read the source files

In [10]:
bloomberg_wb = openpyxl.load_workbook(BLOOMBERG_PATH, read_only=True, data_only=True)
expected_sheets = {"General_Features", "Scope_Time_Series", "Financial_Time_Series"}
missing_sheets = expected_sheets - set(bloomberg_wb.sheetnames)
if missing_sheets:
    raise ValueError(f"Missing Bloomberg sheets: {sorted(missing_sheets)}")

general_security, general_diag = read_bloomberg_sheet(bloomberg_wb["General_Features"])
scope_security, scope_diag = read_bloomberg_sheet(bloomberg_wb["Scope_Time_Series"])
financial_security, financial_diag = read_bloomberg_sheet(bloomberg_wb["Financial_Time_Series"])

In [11]:
panel = pd.read_csv(PANEL_PATH)

required_panel_columns = {
    "company_id", "observation_year", "primary_ticker", "constituent_tickers",
    "security_count", "company_name", "sector",
}
missing_columns = required_panel_columns - set(panel.columns)
if missing_columns:
    raise ValueError(f"Missing panel columns: {sorted(missing_columns)}")
if panel.duplicated(["company_id", "observation_year"]).any():
    raise ValueError("The panel contains duplicate company-year rows.")

companies = (
    panel.sort_values(["company_id", "observation_year"])
    .drop_duplicates("company_id", keep="last")
    [["company_id", "primary_ticker", "constituent_tickers", "security_count", "company_name", "sector"]]
    .sort_values("company_id")
    .reset_index(drop=True)
)

### Resolve company identities

Map Bloomberg securities to the panel company identifiers and collapse verified duplicate share classes.

In [12]:
# Expand pipe-separated share classes into one mapping row per security.
security_map = (
    companies[["company_id", "primary_ticker", "constituent_tickers"]]
    .assign(normalized_ticker=lambda frame: frame["constituent_tickers"].str.split("|", regex=False))
    .explode("normalized_ticker")
    .assign(normalized_ticker=lambda frame: frame["normalized_ticker"].map(normalize_ticker))
    .drop(columns="constituent_tickers")
    .reset_index(drop=True)
)

if not security_map["normalized_ticker"].is_unique:
    raise ValueError("A normalized ticker maps to more than one company.")

In [13]:
def map_and_collapse(frame, sheet_name):
    """Map securities to companies and retain one verified row per company."""
    working = frame.copy()
    working["normalized_ticker"] = working["ID"].map(normalize_ticker)
    working = working.merge(security_map, on="normalized_ticker", how="left", validate="one_to_one")

    if working["company_id"].isna().any():
        missing_ids = working.loc[working["company_id"].isna(), "ID"].tolist()
        raise ValueError(f"Unmapped securities in {sheet_name}: {missing_ids}")

    # Duplicate share classes must agree before one row is retained.
    identity_columns = {
        "ID", "name()", "id_isin()", "gics_sector_name()", "normalized_ticker",
        "company_id", "primary_ticker",
    }
    value_columns = [column for column in working.columns if column not in identity_columns]
    conflicts = []
    for company_id, group in working.groupby("company_id", sort=False):
        if len(group) <= 1:
            continue
        for column in value_columns:
            values = group[column].tolist()
            if any(not values_equal(values[0], value) for value in values[1:]):
                conflicts.append({
                    "sheet": sheet_name,
                    "company_id": company_id,
                    "field": column,
                    "security_values": {row.ID: getattr(row, column) for row in group.itertuples(index=False)},
                })

    # Prefer the panel's primary share class after the agreement check.
    working["is_primary_security"] = working["normalized_ticker"].eq(
        working["primary_ticker"].map(normalize_ticker)
    )
    collapsed = (
        working.sort_values(["company_id", "is_primary_security"], ascending=[True, False])
        .drop_duplicates("company_id", keep="first")
        .sort_values("company_id")
        .reset_index(drop=True)
    )
    return working, collapsed, conflicts

In [14]:
general_mapped, general_company, general_conflicts = map_and_collapse(general_security, "General_Features")
scope_mapped, scope_company, scope_conflicts = map_and_collapse(scope_security, "Scope_Time_Series")
financial_mapped, financial_company, financial_conflicts = map_and_collapse(financial_security, "Financial_Time_Series")

share_class_conflicts = general_conflicts + scope_conflicts + financial_conflicts
if share_class_conflicts:
    raise ValueError(f"Conflicting share-class values: {share_class_conflicts[:5]}")

observed_dual = {
    company_id: set(group["normalized_ticker"])
    for company_id, group in general_mapped.groupby("company_id")
    if len(group) > 1
}

workbook_modified = bloomberg_wb.properties.modified
GENERAL_FEATURES_AS_OF_DATE = workbook_modified.date().isoformat() if workbook_modified else SCORING_AS_OF_DATE
GENERAL_FEATURES_AS_OF_DATE_BASIS = "workbook core modified-date proxy; not a field-effective date"

### Input checks

In [15]:
# Keep the opening check compact; detailed score checks run at the end.
input_checks = {
    "required Bloomberg sheets found": not missing_sheets,
    "500 companies found": len(companies) == 500,
    "503 Bloomberg securities mapped": len(general_mapped) == 503,
    "one panel row per company and year": not panel.duplicated(["company_id", "observation_year"]).any(),
    "panel covers 2012-2024": sorted(panel["observation_year"].unique()) == list(range(2012, 2025)),
    "share-class values agree": not share_class_conflicts,
}
if not all(input_checks.values()):
    raise AssertionError({name: passed for name, passed in input_checks.items() if not passed})

input_summary = pd.Series({
    "Bloomberg securities": len(general_mapped),
    "Companies": len(companies),
    "Panel rows": len(panel),
    "Panel years": f"{panel.observation_year.min()}-{panel.observation_year.max()}",
    "Multiple-share-class companies": len(observed_dual),
}, name="value").to_frame()
display(input_summary)

validation = {
    "assessment_description": ASSESSMENT_DESCRIPTION,
    "input_sha256": {
        "bloomberg_workbook": sha256(BLOOMBERG_PATH),
        "annual_panel": sha256(PANEL_PATH),
    },
    "input_checks": input_checks,
    "sheet_cleaning": {
        "General_Features": general_diag,
        "Scope_Time_Series": scope_diag,
        "Financial_Time_Series": financial_diag,
    },
    "dual_share_classes": {key: sorted(value) for key, value in observed_dual.items()},
}

,value
Bloomberg securities,503
Companies,500
Panel rows,6500
Panel years,2012-2024
Multiple-share-class companies,3


## 3. Build company features

Join current company fields to the 2019–2024 time series, then calculate emissions trends and categorical indicators.

In [16]:
identity = companies.copy()
identity["bloomberg_security_count_mapped"] = identity["company_id"].map(general_mapped.groupby("company_id").size())
identity["bloomberg_security_ids"] = identity["company_id"].map(general_mapped.groupby("company_id")["ID"].apply(lambda s: "|".join(sorted(s))))

def select_features(collapsed, columns, prefix=None):
    out = collapsed[["company_id"] + columns].copy()
    if prefix:
        out = out.rename(columns={column: f"{prefix}{column}" for column in columns})
    return out

general_fields = [c for c in general_security.columns if c not in {"ID", "name()", "id_isin()", "gics_sector_name()"}]
scope_fields = [c for c in scope_security.columns if c not in {"ID", "name()", "id_isin()", "gics_sector_name()"}]
financial_fields = [c for c in financial_security.columns if c not in {"ID", "name()", "id_isin()", "gics_sector_name()"}]

inputs = identity.merge(select_features(general_company, general_fields), on="company_id", how="left", validate="one_to_one")
inputs = inputs.merge(select_features(scope_company, scope_fields), on="company_id", how="left", validate="one_to_one")
inputs = inputs.merge(select_features(financial_company, financial_fields), on="company_id", how="left", validate="one_to_one")

numeric_fields = [c for c in inputs.columns if c.startswith(("GHG_", "SALES_", "BS_", "EBITDA_", "CF_", "NUM_OF_"))]
numeric_fields += [
    "ENERGY_CONSUMPTION", "RENEW_ENERGY_USE", "WATER_CONSUMPTION", "PCT_WATER_RECYCLED",
    "DISCHARGE_TO_WATER", "TOTAL_WASTE", "HAZARDOUS_WASTE", "WASTE_RECYCLED",
    "NUM_ENVIRON_FINES", "ENVIRON_FINES_AMT", "NUMBER_SPILLS", "AMOUNT_OF_SPILLS",
    "PCT_OF_GREEN_SUSTAIN_REVENUE", "WORK_ACCIDENTS_EMPLOYEES", "FATALITIES_CONTRACTORS",
    "FATALITIES_TOTAL", "EMPLOYEE_TRAINING_COST", "COMMUNITY_SPENDING", "EMPLOYEE_TURNOVER_PCT",
    "PCT_WOMEN_EMPLOYEES", "PCT_WOMEN_MGT", "FATALITIES_EMPLOYEES", "LOST_TIME_INCIDENT_RATE",
    "PCT_EMPLOYEES_UNIONIZED", "BOARD_SIZE", "PCT_INDEPENDENT_DIRECTORS",
    "AUDIT_CMTE_INDEPENDENCE_FLD_SCR", "SIZE_OF_AUDIT_COMMITTEE", "BOARD_AVERAGE_TENURE",
    "BOARD_AVERAGE_AGE", "PCT_OF_NON_EXEC_DIR_ON_BRD", "PCT_NON_EXEC_DIR_ON_AUD_CMTE",
    "PCT_NON_EXEC_DIR_ON_CMPNSTN_CMTE", "CHIEF_EXECUTIVE_OFFICER_TENURE",
    "PCT_OF_EXECUTIVES_THAT_ARE_WOMEN", "BOARD_DURATION", "BOARD_MEETINGS_PER_YR",
    "BOARD_MEETING_ATTENDANCE_PCT",
]
for column in sorted(set(numeric_fields).intersection(inputs.columns)):
    inputs[column] = pd.to_numeric(inputs[column], errors="coerce")

In [17]:
def sum_if_complete(left, right):
    """Add two reported values only when both are available."""
    return left + right if pd.notna(left) and pd.notna(right) else np.nan


inputs["scope12_reported_amount_2024"] = [
    sum_if_complete(scope1, scope2)
    for scope1, scope2 in zip(inputs["GHG_SCOPE_1_2024"], inputs["GHG_SCOPE_2_2024"])
]


def annualized_log_trend(years, values, min_observations=4):
    """Estimate annual percentage change from positive, comparable observations."""
    pairs = [(year, value) for year, value in zip(years, values) if pd.notna(value) and float(value) > 0]
    if len(pairs) < min_observations:
        return np.nan, len(pairs)
    x = np.array([year for year, _ in pairs], dtype=float)
    y = np.log(np.array([value for _, value in pairs], dtype=float))
    slope = np.polyfit(x, y, 1)[0]
    return float(np.expm1(slope) * 100), len(pairs)

In [18]:
# Require at least four positive annual observations for each log trend.
absolute_trends = []
employee_intensity_trends = []
absolute_observations = []
intensity_observations = []
for row in inputs.itertuples(index=False):
    scope12 = []
    intensity = []
    for year in TREND_YEARS:
        s1 = getattr(row, f"GHG_SCOPE_1_{year}")
        s2 = getattr(row, f"GHG_SCOPE_2_{year}")
        employees = getattr(row, f"NUM_OF_EMPLOYEES_{year}")
        amount = sum_if_complete(s1, s2)
        scope12.append(amount)
        intensity.append(amount / employees if pd.notna(amount) and pd.notna(employees) and employees > 0 else np.nan)
    abs_trend, abs_n = annualized_log_trend(TREND_YEARS, scope12)
    int_trend, int_n = annualized_log_trend(TREND_YEARS, intensity)
    absolute_trends.append(abs_trend)
    employee_intensity_trends.append(int_trend)
    absolute_observations.append(abs_n)
    intensity_observations.append(int_n)

inputs["scope12_absolute_trend_pct_per_year"] = absolute_trends
inputs["scope12_absolute_trend_observations"] = absolute_observations
inputs["scope12_employee_intensity_trend_pct_per_year"] = employee_intensity_trends
inputs["scope12_employee_intensity_trend_observations"] = intensity_observations

In [19]:
def transition_classification(absolute_trend, intensity_trend):
    """Describe the direction of absolute and employee-adjusted emissions."""
    if pd.isna(absolute_trend) or pd.isna(intensity_trend):
        return "insufficient comparable observations"
    absolute = "absolute ↓" if absolute_trend <= 0 else "absolute ↑"
    intensity = "intensity ↓" if intensity_trend <= 0 else "intensity ↑"
    interpretation = {
        ("absolute ↓", "intensity ↓"): "strong transition",
        ("absolute ↑", "intensity ↓"): "efficiency improvement but growing footprint",
        ("absolute ↓", "intensity ↑"): "falling footprint with deteriorating efficiency",
        ("absolute ↑", "intensity ↑"): "deterioration",
    }[(absolute, intensity)]
    return f"{absolute} and {intensity}: {interpretation}"

inputs["emissions_transition_classification"] = [
    transition_classification(a, i)
    for a, i in zip(inputs["scope12_absolute_trend_pct_per_year"], inputs["scope12_employee_intensity_trend_pct_per_year"])
]

In [20]:
def yn_numeric(series, good_value="Y"):
    """Map a Y/N disclosure field to 1/0 while leaving blanks missing."""
    mapping = {good_value: 1.0, ("N" if good_value == "Y" else "Y"): 0.0}
    return series.astype("string").str.upper().map(mapping).astype(float)

inputs["sbti_status_raw"] = inputs["SBTI_NEAR_TERM_TARGET_STATUS"].map({"Targets Set": 2.0, "Committed": 1.0, "Removed": 0.0})
inputs["climate_governance_support_raw"] = yn_numeric(inputs["CSR_SUSTAINABILITY_COMMITTEE"], "Y")
inputs["ceo_separation_raw"] = yn_numeric(inputs["CEO_DUALITY"], "N")
inputs["sustainability_committee_raw"] = yn_numeric(inputs["CSR_SUSTAINABILITY_COMMITTEE"], "Y")

policy_columns = ["UN_GLOBAL_COMPACT_SIGNATORY", "HUMAN_RIGHTS_POLICY", "EQUAL_OPPORTUNITY_POLICY", "FAIR_REMUNERATION_POLICY", "POLICY_AGAINST_CHILD_LABOR"]
policy_numeric = pd.concat([yn_numeric(inputs[column], "Y").rename(column) for column in policy_columns], axis=1)
inputs["social_policy_composite_raw"] = policy_numeric.mean(axis=1, skipna=True)
inputs["social_policy_fields_observed"] = policy_numeric.notna().sum(axis=1)
inputs["diversity_raw"] = inputs[["PCT_WOMEN_EMPLOYEES", "PCT_WOMEN_MGT"]].mean(axis=1, skipna=True)
inputs["diversity_fields_observed"] = inputs[["PCT_WOMEN_EMPLOYEES", "PCT_WOMEN_MGT"]].notna().sum(axis=1)

In [21]:
potential_conditions = {
    "scope12_revenue_intensity": inputs["scope12_reported_amount_2024"].notna() & inputs["SALES_REV_TURN_2024"].gt(0),
    "scope3_revenue_intensity": inputs["GHG_SCOPE_3_2024"].notna() & inputs["SALES_REV_TURN_2024"].gt(0),
    "energy_revenue_intensity": inputs["ENERGY_CONSUMPTION"].notna() & inputs["SALES_REV_TURN_2024"].gt(0),
    "resource_revenue_intensity": inputs[["WATER_CONSUMPTION", "TOTAL_WASTE"]].notna().any(axis=1) & inputs["SALES_REV_TURN_2024"].gt(0),
    "renewable_energy_ratio": inputs["RENEW_ENERGY_USE"].notna() & inputs["ENERGY_CONSUMPTION"].gt(0),
}

print("2024 Scope 1+2 reported-amount coverage:", f"{inputs['scope12_reported_amount_2024'].notna().mean():.1%}")
print("Comparable absolute trends (>=4 years):", inputs["scope12_absolute_trend_pct_per_year"].notna().sum())
print("Comparable per-employee intensity trends (>=4 years):", inputs["scope12_employee_intensity_trend_pct_per_year"].notna().sum())
display(inputs[["company_id", "primary_ticker", "sector", "scope12_reported_amount_2024", "scope12_absolute_trend_pct_per_year", "scope12_employee_intensity_trend_pct_per_year", "emissions_transition_classification"]].head())

2024 Scope 1+2 reported-amount coverage: 90.6%
Comparable absolute trends (>=4 years): 436
Comparable per-employee intensity trends (>=4 years): 420


,company_id,primary_ticker,sector,scope12_reported_amount_2024,scope12_absolute_trend_pct_per_year,scope12_employee_intensity_trend_pct_per_year,emissions_transition_classification
0,ivv:A,A,Health Care,79.696001,0.123225,NaN,insufficient comparable observations
1,ivv:AAPL,AAPL,Information Technology,1279.700001,7.792075,4.054114,absolute ↑ and intensity ↑: deterioration
2,ivv:ABBV,ABBV,Health Care,541.213989,-2.506201,-11.066016,absolute ↓ and intensity ↓: strong transition
3,ivv:ABNB,ABNB,Consumer Discretionary,6.475000,0.989444,-0.588265,absolute ↑ and intensity ↓: efficiency improve...
4,ivv:ABT,ABT,Health Care,974.000000,-1.056225,-2.373072,absolute ↓ and intensity ↓: strong transition


### Review feature coverage

In [22]:
candidate_specs = [
    dict(pillar="Environmental", feature="scope12_revenue_intensity", source_field="GHG_SCOPE_1_2024 + GHG_SCOPE_2_2024 / SALES_REV_TURN_2024", description="Scope 1+2 emissions per revenue", unit="Not calculated: emissions physical unit and sales reporting currency absent", direction="adverse", time_basis="2024", numerator="GHG_SCOPE_1_2024 + GHG_SCOPE_2_2024", denominator="SALES_REV_TURN_2024", missing_rule="Missing remains missing", applicability_rule="Sector materiality matrix", status="Disabled", reason="Failed unit/currency comparability check"),
    dict(pillar="Environmental", feature="scope3_revenue_intensity", source_field="GHG_SCOPE_3_2024 / SALES_REV_TURN_2024", description="Scope 3 emissions per revenue", unit="Not calculated: emissions physical unit and sales reporting currency absent", direction="adverse", time_basis="2024", numerator="GHG_SCOPE_3_2024", denominator="SALES_REV_TURN_2024", missing_rule="Missing remains missing", applicability_rule="Sector materiality matrix", status="Disabled", reason="Failed unit/currency comparability check; lower coverage"),
    dict(pillar="Environmental", feature="energy_revenue_intensity", source_field="ENERGY_CONSUMPTION / SALES_REV_TURN_2024", description="Energy use per revenue", unit="Not calculated: energy unit, aligned period, and sales reporting currency absent", direction="adverse", time_basis="Mixed candidate; invalid", numerator="ENERGY_CONSUMPTION", denominator="SALES_REV_TURN_2024", missing_rule="Missing remains missing", applicability_rule="Sector materiality matrix", status="Disabled", reason="Failed unit, period, and currency checks"),
    dict(pillar="Environmental", feature="resource_revenue_intensity", source_field="WATER_CONSUMPTION; TOTAL_WASTE / SALES_REV_TURN_2024", description="Resource use per revenue", unit="Not calculated: incompatible resource units and sales currency absent", direction="adverse", time_basis="Mixed candidate; invalid", numerator="Water/waste candidate fields", denominator="SALES_REV_TURN_2024", missing_rule="Missing remains missing", applicability_rule="Sector materiality matrix", status="Disabled", reason="No coherent verified numerator; failed currency check"),
    dict(pillar="Environmental", feature="renewable_energy_ratio", source_field="RENEW_ENERGY_USE / ENERGY_CONSUMPTION", description="Renewable share of energy use", unit="Not calculated: exact source units and aligned period absent", direction="beneficial", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="RENEW_ENERGY_USE", denominator="ENERGY_CONSUMPTION", missing_rule="Missing remains missing", applicability_rule="Sector materiality matrix", status="Disabled", reason="Exact numerator/denominator units and period not documented"),
    dict(pillar="Environmental", feature="scope12_reported_amount_2024", source_field="GHG_SCOPE_1_2024 + GHG_SCOPE_2_2024", description="Reported Scope 1+2 footprint, used only as an ordinal outcome", unit="Bloomberg native reported amount; exact physical unit not supplied", direction="adverse", time_basis="2024", numerator="GHG_SCOPE_1_2024 + GHG_SCOPE_2_2024", denominator="None", missing_rule="Missing if either scope is missing", applicability_rule="Sector materiality matrix", status="Enabled", reason="Same standardized fields support ordinal sector/global percentiles; no physical-unit claim"),
    dict(pillar="Transition", feature="scope12_employee_intensity_trend", source_field="GHG_SCOPE_1/2_2019:2024 and NUM_OF_EMPLOYEES_2019:2024", description="Annualized trend in reported Scope 1+2 amount per employee", unit="percent per year; unknown emissions scale cancels", direction="adverse", time_basis="2019-2024; >=4 comparable years", numerator="Annual Scope 1+2 reported amount", denominator="Annual employee count", missing_rule="Missing with fewer than four positive comparable pairs", applicability_rule="Sector materiality matrix", status="Enabled", reason="Within-company log trend is scale-invariant; employee denominator is a count"),
    dict(pillar="Transition", feature="scope12_absolute_trend", source_field="GHG_SCOPE_1/2_2019:2024", description="Annualized trend in absolute reported Scope 1+2 amount", unit="percent per year; unknown emissions scale cancels", direction="adverse", time_basis="2019-2024; >=4 comparable years", numerator="Annual Scope 1+2 reported amount", denominator="None", missing_rule="Missing with fewer than four positive comparable observations", applicability_rule="Sector materiality matrix", status="Enabled", reason="Within-company log trend is scale-invariant"),
    dict(pillar="Transition", feature="sbti_status", source_field="SBTI_NEAR_TERM_TARGET_STATUS", description="SBTi near-term target status", unit="ordinal category", direction="beneficial", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Not applicable", denominator="Not applicable", missing_rule="Missing is not a failed commitment", applicability_rule="Sector materiality matrix", status="Enabled", reason="Verified categorical commitment field"),
    dict(pillar="Transition", feature="climate_governance_support", source_field="CSR_SUSTAINABILITY_COMMITTEE", description="Sustainability committee as climate-governance support proxy", unit="Y/N", direction="beneficial", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Not applicable", denominator="Not applicable", missing_rule="Missing is unscored", applicability_rule="Sector materiality matrix", status="Enabled", reason="Broad governance proxy; not treated as realized climate performance"),
    dict(pillar="Social", feature="diversity", source_field="PCT_WOMEN_EMPLOYEES; PCT_WOMEN_MGT", description="Average observed workforce and management gender representation", unit="percent", direction="beneficial", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Reported percentages", denominator="Reported workforce populations", missing_rule="Average observed subfields only", applicability_rule="Sector materiality matrix", status="Enabled", reason="Direct representation measures with explicit percent units"),
    dict(pillar="Social", feature="employee_safety", source_field="WORK_ACCIDENTS_EMPLOYEES; FATALITIES_EMPLOYEES", description="Employee safety evidence from reported event counts", unit="reported counts", direction="adverse", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Reported employee events", denominator="None aligned; therefore downweighted", missing_rule="Average observed percentile subfields only", applicability_rule="Sector materiality matrix", status="Enabled - downweighted", reason="Useful evidence but no aligned exposure denominator"),
    dict(pillar="Social", feature="social_policy", source_field="Five Y/N policy fields", description="Observed social-policy composite", unit="share of observed Y/N fields", direction="beneficial", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Positive observed policies", denominator="Observed policy fields only", missing_rule="Missing fields excluded; no all-missing score", applicability_rule="Sector materiality matrix", status="Enabled", reason="Transparent policy breadth measure"),
    dict(pillar="Social", feature="employee_stability", source_field="EMPLOYEE_TURNOVER_PCT", description="Employee turnover", unit="percent", direction="adverse", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Reported leavers", denominator="Reported workforce basis", missing_rule="Missing is unscored", applicability_rule="Sector materiality matrix", status="Enabled", reason="Explicit percentage; coverage is reported"),
    dict(pillar="Governance", feature="board_independence", source_field="PCT_INDEPENDENT_DIRECTORS", description="Independent directors", unit="percent", direction="beneficial", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Independent directors", denominator="Board members", missing_rule="Missing is unscored", applicability_rule="Universal", status="Enabled", reason="Direct governance outcome"),
    dict(pillar="Governance", feature="ceo_separation", source_field="CEO_DUALITY", description="CEO and chair roles separated", unit="Y/N transformed so N duality is beneficial", direction="beneficial", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Not applicable", denominator="Not applicable", missing_rule="Missing is unscored", applicability_rule="Universal", status="Enabled", reason="Direct governance structure"),
    dict(pillar="Governance", feature="board_attendance", source_field="BOARD_MEETING_ATTENDANCE_PCT", description="Board meeting attendance", unit="percent", direction="beneficial", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Attended meetings", denominator="Applicable board meetings", missing_rule="Missing is unscored", applicability_rule="Universal", status="Enabled", reason="Direct board-function measure"),
    dict(pillar="Governance", feature="women_executives", source_field="PCT_OF_EXECUTIVES_THAT_ARE_WOMEN", description="Women among executives", unit="percent", direction="beneficial", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Women executives", denominator="Executives", missing_rule="Missing is unscored", applicability_rule="Universal", status="Enabled", reason="Direct leadership-diversity measure"),
    dict(pillar="Governance", feature="sustainability_committee", source_field="CSR_SUSTAINABILITY_COMMITTEE", description="Board/company sustainability committee", unit="Y/N", direction="beneficial", time_basis=GENERAL_FEATURES_TIME_BASIS, numerator="Not applicable", denominator="Not applicable", missing_rule="Missing is unscored", applicability_rule="Universal", status="Enabled", reason="Oversight structure; not a realized outcome"),
]

In [23]:
audit_series = {
    "scope12_reported_amount_2024": inputs["scope12_reported_amount_2024"],
    "scope12_employee_intensity_trend": inputs["scope12_employee_intensity_trend_pct_per_year"],
    "scope12_absolute_trend": inputs["scope12_absolute_trend_pct_per_year"],
    "sbti_status": inputs["sbti_status_raw"],
    "climate_governance_support": inputs["climate_governance_support_raw"],
    "diversity": inputs["diversity_raw"],
    "employee_safety": inputs[["WORK_ACCIDENTS_EMPLOYEES", "FATALITIES_EMPLOYEES"]].mean(axis=1, skipna=True),
    "social_policy": inputs["social_policy_composite_raw"],
    "employee_stability": inputs["EMPLOYEE_TURNOVER_PCT"],
    "board_independence": inputs["PCT_INDEPENDENT_DIRECTORS"],
    "ceo_separation": inputs["ceo_separation_raw"],
    "board_attendance": inputs["BOARD_MEETING_ATTENDANCE_PCT"],
    "women_executives": inputs["PCT_OF_EXECUTIVES_THAT_ARE_WOMEN"],
    "sustainability_committee": inputs["sustainability_committee_raw"],
}
audit_frame = pd.DataFrame(audit_series)
correlations = audit_frame.corr(method="spearman", min_periods=20)
redundant_pairs = []
for i, left in enumerate(correlations.columns):
    for right in correlations.columns[i + 1:]:
        rho = correlations.loc[left, right]
        if pd.notna(rho) and abs(rho) >= HIGH_REDUNDANCY_SPEARMAN:
            redundant_pairs.append({"feature_1": left, "feature_2": right, "spearman": float(rho)})

In [24]:
dictionary_rows = []
for spec in candidate_specs:
    feature = spec["feature"]
    series = audit_series.get(feature)
    if series is not None:
        coverage = float(series.notna().mean())
        variance = float(pd.to_numeric(series, errors="coerce").var()) if series.notna().sum() > 1 else np.nan
    elif feature in potential_conditions:
        coverage = float(potential_conditions[feature].mean())
        variance = np.nan
    else:
        coverage = np.nan
        variance = np.nan
    original_weight = ORIGINAL_FEATURE_WEIGHTS.get(spec["pillar"], {}).get(feature, 0.0)
    final_weight = FINAL_FEATURE_WEIGHTS.get(spec["pillar"], {}).get(feature, 0.0)
    flags = []
    if pd.notna(coverage) and coverage < SPARSE_COVERAGE_THRESHOLD:
        flags.append("very sparse")
    if pd.notna(variance) and np.isclose(variance, 0):
        flags.append("zero variance")
    if any(feature in (pair["feature_1"], pair["feature_2"]) for pair in redundant_pairs):
        flags.append("highly redundant")
    dictionary_rows.append({
        "pillar": spec["pillar"], "output_feature": feature, "source_field": spec["source_field"],
        "description": spec["description"], "unit": spec["unit"], "direction": spec["direction"],
        "time_basis": spec["time_basis"], "numerator": spec["numerator"], "denominator": spec["denominator"],
        "missing_value_rule": spec["missing_rule"], "applicability_rule": spec["applicability_rule"],
        "overall_coverage": coverage, "variance": variance, "audit_flags": "; ".join(flags) if flags else "none",
        "feature_status": spec["status"], "decision_reason": spec["reason"],
        "original_weight": original_weight, "final_weight": final_weight,
        "economic_or_environmental_interpretation": spec["description"],
    })

feature_dictionary = pd.DataFrame(dictionary_rows)
coverage_rows = []
coverage_series = {**audit_series, **potential_conditions}
for feature, series in coverage_series.items():
    observed_mask = series.astype(bool) if pd.api.types.is_bool_dtype(series) else series.notna()
    for sector, idx in inputs.groupby("sector").groups.items():
        coverage_rows.append({"feature": feature, "sector": sector, "coverage": float(observed_mask.loc[idx].mean()), "company_count": len(idx)})
feature_coverage_by_sector = pd.DataFrame(coverage_rows)

display(feature_dictionary[["pillar", "output_feature", "unit", "overall_coverage", "feature_status", "final_weight"]])

,pillar,output_feature,unit,overall_coverage,feature_status,final_weight
0,Environmental,scope12_revenue_intensity,Not calculated: emissions physical unit and sa...,0.906,Disabled,0.00
1,Environmental,scope3_revenue_intensity,Not calculated: emissions physical unit and sa...,0.840,Disabled,0.00
2,Environmental,energy_revenue_intensity,"Not calculated: energy unit, aligned period, a...",0.876,Disabled,0.00
3,Environmental,resource_revenue_intensity,Not calculated: incompatible resource units an...,0.446,Disabled,0.00
4,Environmental,renewable_energy_ratio,Not calculated: exact source units and aligned...,0.354,Disabled,0.00
5,Environmental,scope12_reported_amount_2024,Bloomberg native reported amount; exact physic...,0.906,Enabled,1.00
6,Transition,scope12_employee_intensity_trend,percent per year; unknown emissions scale cancels,0.840,Enabled,0.50
7,Transition,scope12_absolute_trend,percent per year; unknown emissions scale cancels,0.872,Enabled,0.00
8,Transition,sbti_status,ordinal category,0.512,Enabled,0.35
9,Transition,climate_governance_support,Y/N,0.940,Enabled,0.15


### Normalize features

Convert observed values to 0–100 peer scores and record the peer group used for each feature.

In [25]:
# Use sector peers when the group is large enough; otherwise use the full universe.
def percentile_good_scores(frame, value_column, beneficial, min_peers=MIN_PEER_COUNT):
    """Return comparable 0-100 scores and the peer group used."""
    values = pd.to_numeric(frame[value_column], errors="coerce")
    global_score = values.rank(method="average", pct=True) * 100
    if not beneficial:
        global_score = 100 - global_score
    sector_score = pd.Series(np.nan, index=frame.index, dtype=float)
    peer_group = pd.Series(pd.NA, index=frame.index, dtype="string")
    peer_count = pd.Series(pd.NA, index=frame.index, dtype="Int64")
    fallback = pd.Series(False, index=frame.index, dtype=bool)
    global_n = int(values.notna().sum())
    for sector, idx in frame.groupby("sector").groups.items():
        observed_idx = [i for i in idx if pd.notna(values.loc[i])]
        sector_n = len(observed_idx)
        if sector_n >= min_peers:
            ranked = values.loc[observed_idx].rank(method="average", pct=True) * 100
            if not beneficial:
                ranked = 100 - ranked
            sector_score.loc[observed_idx] = ranked
            peer_group.loc[observed_idx] = f"sector:{sector}"
            peer_count.loc[observed_idx] = sector_n
        else:
            sector_score.loc[observed_idx] = global_score.loc[observed_idx]
            peer_group.loc[observed_idx] = "global fallback"
            peer_count.loc[observed_idx] = global_n
            fallback.loc[observed_idx] = True
    return pd.DataFrame({
        "sector_score": sector_score.clip(0, 100), "global_score": global_score.clip(0, 100),
        "peer_group": peer_group, "peer_count": peer_count, "fallback_used": fallback,
    })

In [26]:
feature_score_specs = {
    "scope12_reported_amount_2024": ("scope12_reported_amount_2024", False),
    "scope12_employee_intensity_trend": ("scope12_employee_intensity_trend_pct_per_year", False),
    "scope12_absolute_trend": ("scope12_absolute_trend_pct_per_year", False),
    "sbti_status": ("sbti_status_raw", True),
    "climate_governance_support": ("climate_governance_support_raw", True),
    "diversity": ("diversity_raw", True),
    "work_accidents": ("WORK_ACCIDENTS_EMPLOYEES", False),
    "employee_fatalities": ("FATALITIES_EMPLOYEES", False),
    "social_policy": ("social_policy_composite_raw", True),
    "employee_stability": ("EMPLOYEE_TURNOVER_PCT", False),
    "board_independence": ("PCT_INDEPENDENT_DIRECTORS", True),
    "ceo_separation": ("ceo_separation_raw", True),
    "board_attendance": ("BOARD_MEETING_ATTENDANCE_PCT", True),
    "women_executives": ("PCT_OF_EXECUTIVES_THAT_ARE_WOMEN", True),
    "sustainability_committee": ("sustainability_committee_raw", True),
}

for feature, (raw_column, beneficial) in feature_score_specs.items():
    result = percentile_good_scores(inputs, raw_column, beneficial)
    inputs[f"{feature}_sector_score"] = result["sector_score"]
    inputs[f"{feature}_global_score"] = result["global_score"]
    inputs[f"{feature}_peer_group"] = result["peer_group"]
    inputs[f"{feature}_peer_count"] = result["peer_count"]
    inputs[f"{feature}_peer_fallback_used"] = result["fallback_used"]
    inputs[f"{feature}_score"] = result["sector_score"]

inputs["scope12_reported_amount_2024_score"] = (
    ENVIRONMENT_SECTOR_MIX_LAMBDA * inputs["scope12_reported_amount_2024_sector_score"]
    + (1 - ENVIRONMENT_SECTOR_MIX_LAMBDA) * inputs["scope12_reported_amount_2024_global_score"]
)

In [27]:
inputs["employee_safety_score"] = inputs[["work_accidents_score", "employee_fatalities_score"]].mean(axis=1, skipna=True)
inputs["employee_safety_peer_group"] = np.where(
    inputs[["work_accidents_score", "employee_fatalities_score"]].notna().any(axis=1),
    "composite of observed sector/fallback percentiles", pd.NA,
)
inputs["employee_safety_peer_count"] = inputs[["work_accidents_peer_count", "employee_fatalities_peer_count"]].min(axis=1, skipna=True).astype("Int64")

peer_columns = [c for c in inputs.columns if c.endswith("_peer_group")]
peer_diagnostics = []
for column in peer_columns:
    feature = column.removesuffix("_peer_group")
    peer_diagnostics.append({
        "feature": feature,
        "observed": int(inputs[column].notna().sum()),
        "global_fallback_count": int(inputs[column].eq("global fallback").sum()),
        "minimum_peer_count_used": int(inputs[column.replace("_peer_group", "_peer_count")].min()) if inputs[column.replace("_peer_group", "_peer_count")].notna().any() else None,
    })
peer_diagnostics = pd.DataFrame(peer_diagnostics)
display(peer_diagnostics)

,feature,observed,global_fallback_count,minimum_peer_count_used
0,scope12_reported_amount_2024,453,13,19
1,scope12_employee_intensity_trend,420,11,19
2,scope12_absolute_trend,436,11,19
3,sbti_status,256,32,16
4,climate_governance_support,470,0,17
5,diversity,458,14,21
6,work_accidents,153,45,16
7,employee_fatalities,174,19,15
8,social_policy,436,0,17
9,employee_stability,142,39,15


## 4. Calculate company scores

Reweight observed features within each block and shrink low-coverage results toward the neutral score of 50.

In [28]:
def confidence_grade(weight):
    """Translate observed evidence weight into a coverage grade."""
    if weight >= CONFIDENCE_THRESHOLDS["High"]:
        return "High"
    if weight >= CONFIDENCE_THRESHOLDS["Medium"]:
        return "Medium"
    return "Low"


def calculate_block(frame, feature_weights, materiality, threshold=SHRINKAGE_EVIDENCE_THRESHOLD, prior=NEUTRAL_PRIOR):
    """Calculate one score block from observed, materiality-weighted features."""
    raw_scores = []
    observed_weights = []
    applicable_weights = []

    for row in frame.itertuples(index=False):
        weighted_sum = 0.0
        observed_total = 0.0
        applicable_total = 0.0

        for feature, configured_weight in feature_weights.items():
            multiplier = materiality.get(feature, {}).get(row.sector, 1.0)
            applicable_weight = configured_weight * multiplier
            if applicable_weight <= 0:
                continue

            applicable_total += applicable_weight
            score = getattr(row, f"{feature}_score")
            if pd.notna(score):
                weighted_sum += applicable_weight * float(score)
                observed_total += applicable_weight

        raw_scores.append(weighted_sum / observed_total if observed_total else np.nan)
        observed_weights.append(observed_total / applicable_total if applicable_total else 0.0)
        applicable_weights.append(applicable_total)

    raw = pd.Series(raw_scores, index=frame.index, dtype=float)
    observed = pd.Series(observed_weights, index=frame.index, dtype=float).clip(0, 1)
    prior_series = (
        pd.Series(float(prior), index=frame.index)
        if np.isscalar(prior)
        else pd.Series(prior, index=frame.index).fillna(NEUTRAL_PRIOR)
    )

    # Sparse blocks move only part of the way from the prior to the raw score.
    evidence_factor = np.minimum(1.0, observed / threshold)
    adjusted = prior_series + evidence_factor * (raw.fillna(prior_series) - prior_series)

    return pd.DataFrame({
        "raw": raw,
        "adjusted": adjusted.clip(0, 100),
        "observed_weight": observed,
        "applicable_weight": applicable_weights,
        "confidence": observed.map(confidence_grade),
    })

In [29]:
PILLAR_FEATURE_KEYS = {
    "environmental": "Environmental", "transition": "Transition", "social": "Social", "governance": "Governance"
}
for prefix, title in PILLAR_FEATURE_KEYS.items():
    result = calculate_block(inputs, FINAL_FEATURE_WEIGHTS[title], FEATURE_MATERIALITY)
    for column in result.columns:
        inputs[f"{prefix}_{column}"] = result[column]

inputs["structured_score_before_regulatory_penalty"] = sum(
    FINAL_PILLAR_WEIGHTS[title] * inputs[f"{prefix}_adjusted"]
    for prefix, title in PILLAR_FEATURE_KEYS.items()
)

In [30]:
net_zero_result = calculate_block(inputs, FINAL_NET_ZERO_WEIGHTS, FEATURE_MATERIALITY)
inputs["net_zero_transition_raw_score"] = net_zero_result["raw"]
inputs["net_zero_transition_score"] = net_zero_result["adjusted"]
inputs["net_zero_transition_observed_weight"] = net_zero_result["observed_weight"]
inputs["net_zero_transition_confidence"] = net_zero_result["confidence"]

commitment_result = calculate_block(inputs, COMMITMENT_WEIGHTS, FEATURE_MATERIALITY)
inputs["commitment_score"] = commitment_result["adjusted"]
inputs["commitment_observed_weight"] = commitment_result["observed_weight"]
inputs["commitment_confidence"] = commitment_result["confidence"]

In [31]:
for prefix in PILLAR_FEATURE_KEYS:
    assert inputs[f"{prefix}_adjusted"].between(0, 100).all()
assert inputs["structured_score_before_regulatory_penalty"].between(0, 100).all()
assert inputs["net_zero_transition_score"].between(0, 100).all()

pillar_summary = pd.DataFrame([
    {
        "pillar": title,
        "configured_weight": FINAL_PILLAR_WEIGHTS[title],
        "raw_min": inputs[f"{prefix}_raw"].min(), "raw_max": inputs[f"{prefix}_raw"].max(),
        "adjusted_min": inputs[f"{prefix}_adjusted"].min(), "adjusted_max": inputs[f"{prefix}_adjusted"].max(),
        "mean_observed_weight": inputs[f"{prefix}_observed_weight"].mean(),
        "low_confidence_companies": inputs[f"{prefix}_confidence"].eq("Low").sum(),
    }
    for prefix, title in PILLAR_FEATURE_KEYS.items()
])
display(pillar_summary.round(3))

,pillar,configured_weight,raw_min,raw_max,adjusted_min,adjusted_max,mean_observed_weight,low_confidence_companies
0,Environmental,0.45,0.000,98.981,0.000,98.981,0.906,47
1,Transition,0.15,3.468,87.904,3.468,85.196,0.728,64
2,Social,0.20,1.429,98.571,2.674,91.758,0.766,72
3,Governance,0.20,3.006,91.068,13.711,87.548,0.941,6


### Add regulatory evidence

Aggregate 2020–2024 source evidence, calculate a separate deduction, and update company ranks.

In [32]:
# Give more weight to recent observations.
recent_panel = panel.loc[panel["observation_year"].isin(REGULATORY_YEARS)].copy()
recent_panel["recency_weight"] = recent_panel["observation_year"].map(REGULATORY_RECENCY_WEIGHTS)

def recency_weighted_metric(group, metric, match_flag, additional_eligibility=None):
    """Summarize one company's observed regulatory values across recent years."""
    eligible = group[match_flag].eq(1) & group[metric].notna()
    if additional_eligibility is not None:
        eligible &= additional_eligibility(group)
    observed = group.loc[eligible]
    if observed.empty:
        return pd.Series({"value": np.nan, "years_observed": 0, "latest_observed_year": np.nan})
    value = np.average(observed[metric].astype(float), weights=observed["recency_weight"].astype(float))
    return pd.Series({"value": float(value), "years_observed": len(observed), "latest_observed_year": int(observed["observation_year"].max())})

In [33]:
reg_specs = {
    "tri": ("tri_total_releases_lb_per_facility", "tri_match_flag", None),
    "cfpb": ("cfpb_not_timely_response_rate", "cfpb_match_flag", lambda g: g["cfpb_complaints"].fillna(0).ge(10)),
    "cpsc": ("cpsc_named_recall_count", "cpsc_match_flag", None),
    "openfda": ("openfda_enforcement_event_count", "openfda_match_flag", None),
}
reg_aggregates = companies[["company_id"]].copy()
for source, (metric, flag, eligibility) in reg_specs.items():
    aggregate = (
        recent_panel.groupby("company_id", sort=False)
        .apply(lambda group: recency_weighted_metric(group, metric, flag, eligibility), include_groups=False)
        .rename(columns={"value": f"{source}_recent_value", "years_observed": f"{source}_recent_years_observed", "latest_observed_year": f"{source}_latest_observed_year"})
        .reset_index()
    )
    reg_aggregates = reg_aggregates.merge(aggregate, on="company_id", how="left", validate="one_to_one")

inputs = inputs.merge(reg_aggregates, on="company_id", how="left", validate="one_to_one")
for source in reg_specs:
    temp_column = f"{source}_recent_value"
    ranked = percentile_good_scores(inputs, temp_column, beneficial=False, min_peers=MIN_REGULATORY_PEER_COUNT)
    inputs[f"{source}_regulatory_good_score"] = ranked["sector_score"]
    inputs[f"{source}_regulatory_adverse_score"] = 100 - ranked["sector_score"]
    inputs[f"{source}_regulatory_peer_group"] = ranked["peer_group"]
    inputs[f"{source}_regulatory_peer_count"] = ranked["peer_count"]
    inputs[f"{source}_regulatory_peer_fallback_used"] = ranked["fallback_used"]

In [34]:
# Combine only the regulatory sources that apply to the company sector.
penalties = []
penalty_observed_weights = []
penalty_status = []
for row in inputs.itertuples(index=False):
    applicable_total = 0.0
    observed_total = 0.0
    adverse_sum = 0.0
    observed_sources = []
    for source, configured_weight in REGULATORY_SOURCE_WEIGHTS.items():
        materiality = REGULATORY_MATERIALITY[source].get(row.sector, 0.0)
        source_weight = configured_weight * materiality
        if source_weight <= 0:
            continue
        applicable_total += source_weight
        adverse_score = getattr(row, f"{source}_regulatory_adverse_score")
        if pd.notna(adverse_score):
            observed_total += source_weight
            adverse_sum += source_weight * float(adverse_score)
            observed_sources.append(source)
    observed_fraction = observed_total / applicable_total if applicable_total > 0 else 0.0
    if observed_total > 0:
        composite_adverse = adverse_sum / observed_total
        penalty = min(REGULATORY_PENALTY_CAP, REGULATORY_PENALTY_CAP * composite_adverse / 100)
        status = "Observed applicable evidence: " + ", ".join(observed_sources)
    else:
        penalty = 0.0
        status = "No observed applicable regulatory evidence; zero deduction is not a good-performance signal"
    penalties.append(penalty)
    penalty_observed_weights.append(observed_fraction)
    penalty_status.append(status)

In [35]:
inputs["regulatory_evidence_penalty_provisional"] = penalties
inputs["regulatory_penalty_observed_weight"] = penalty_observed_weights
inputs["regulatory_evidence_status"] = penalty_status
inputs["structured_score_after_regulatory_penalty"] = np.maximum(
    0, inputs["structured_score_before_regulatory_penalty"] - inputs["regulatory_evidence_penalty_provisional"]
)
inputs["rank_before_regulatory"] = inputs["structured_score_before_regulatory_penalty"].rank(method="min", ascending=False).astype(int)
inputs["rank_after_regulatory"] = inputs["structured_score_after_regulatory_penalty"].rank(method="min", ascending=False).astype(int)
inputs["rank_delta"] = inputs["rank_before_regulatory"] - inputs["rank_after_regulatory"]

# TRI pollution evidence enters only this provisional penalty and the separate credibility diagnostic, not a sustainability pillar.
inputs["tri_pollution_outcome_score"] = inputs["tri_regulatory_good_score"]
realized_result = calculate_block(inputs, REALIZED_TRANSITION_WEIGHTS, FEATURE_MATERIALITY)
inputs["realized_transition_score"] = realized_result["adjusted"]
inputs["realized_transition_observed_weight"] = realized_result["observed_weight"]
inputs["realized_transition_confidence"] = realized_result["confidence"]
inputs["credibility_gap"] = inputs["commitment_score"] - inputs["realized_transition_score"]

assert inputs["regulatory_evidence_penalty_provisional"].between(0, REGULATORY_PENALTY_CAP).all()
assert inputs["structured_score_after_regulatory_penalty"].between(0, 100).all()

In [36]:
largest_penalties = inputs.nlargest(12, "regulatory_evidence_penalty_provisional")[[
    "primary_ticker", "company_name", "sector", "regulatory_evidence_penalty_provisional",
    "regulatory_penalty_observed_weight", "tri_recent_value", "cfpb_recent_value",
    "cpsc_recent_value", "openfda_recent_value", "rank_delta", "regulatory_evidence_status",
]]
largest_rank_changes = inputs.reindex(inputs["rank_delta"].abs().sort_values(ascending=False).index).head(15)[[
    "primary_ticker", "company_name", "sector", "rank_before_regulatory", "rank_after_regulatory",
    "rank_delta", "regulatory_evidence_penalty_provisional", "regulatory_evidence_status",
]]

display(largest_rank_changes.head(5))

,primary_ticker,company_name,sector,rank_before_regulatory,rank_after_regulatory,rank_delta,regulatory_evidence_penalty_provisional,regulatory_evidence_status
391,ROP,ROPER TECHNOLOGIES,Information Technology,104,263,-159,15.000000,Observed applicable evidence: tri
430,TGT,TARGET CORP,Consumer Staples,207,354,-147,15.000000,Observed applicable evidence: cpsc
403,SNA,SNAP ON INC,Industrials,185,322,-137,12.857143,Observed applicable evidence: tri
225,HOOD,ROBINHOOD MARKETS CLASS A,Financials,41,172,-131,15.000000,Observed applicable evidence: cfpb
361,PHM,PULTEGROUP,Consumer Discretionary,208,336,-128,12.500000,Observed applicable evidence: cfpb


### Build the final company table

In [37]:
inputs["overall_observed_weight"] = sum(
    FINAL_PILLAR_WEIGHTS[title] * inputs[f"{prefix}_observed_weight"]
    for prefix, title in PILLAR_FEATURE_KEYS.items()
)
for prefix in PILLAR_FEATURE_KEYS:
    inputs[f"{prefix}_raw_score"] = inputs[f"{prefix}_raw"]
    inputs[f"{prefix}_adjusted_score"] = inputs[f"{prefix}_adjusted"]
inputs["overall_coverage_grade"] = inputs["overall_observed_weight"].map(confidence_grade)
inputs["confidence_status"] = inputs["overall_coverage_grade"].map({
    "High": "High",
    "Medium": "Medium",
    "Low": "Low - diagnostic only; judge-facing ranking warning",
})
inputs["quantitative_anchor_year"] = QUANTITATIVE_ANCHOR_YEAR
inputs["general_features_time_basis"] = GENERAL_FEATURES_TIME_BASIS
inputs["general_features_as_of_date"] = GENERAL_FEATURES_AS_OF_DATE
inputs["general_features_as_of_date_basis"] = GENERAL_FEATURES_AS_OF_DATE_BASIS
inputs["scoring_as_of_date"] = SCORING_AS_OF_DATE
inputs["assessment_description"] = ASSESSMENT_DESCRIPTION

In [38]:
required_output_columns = [
    "company_id", "primary_ticker", "company_name", "sector", "quantitative_anchor_year",
    "general_features_time_basis", "general_features_as_of_date", "scoring_as_of_date",
    "environmental_raw_score", "environmental_adjusted_score", "environmental_observed_weight",
    "transition_raw_score", "transition_adjusted_score", "transition_observed_weight",
    "social_raw_score", "social_adjusted_score", "social_observed_weight",
    "governance_raw_score", "governance_adjusted_score", "governance_observed_weight",
    "structured_score_before_regulatory_penalty", "regulatory_evidence_penalty_provisional",
    "structured_score_after_regulatory_penalty", "net_zero_transition_score", "commitment_score",
    "realized_transition_score", "credibility_gap", "overall_observed_weight", "overall_coverage_grade",
    "confidence_status", "rank_before_regulatory", "rank_after_regulatory", "rank_delta",
]
extra_output_columns = [
    "constituent_tickers", "security_count", "assessment_description", "general_features_as_of_date_basis",
    "environmental_confidence", "transition_confidence", "social_confidence", "governance_confidence",
    "net_zero_transition_raw_score", "net_zero_transition_observed_weight", "net_zero_transition_confidence",
    "commitment_observed_weight", "commitment_confidence", "realized_transition_observed_weight",
    "realized_transition_confidence", "regulatory_penalty_observed_weight", "regulatory_evidence_status",
    "emissions_transition_classification", "scope12_absolute_trend_pct_per_year",
    "scope12_employee_intensity_trend_pct_per_year",
]
scores = inputs[required_output_columns + extra_output_columns].sort_values("company_id").reset_index(drop=True)
assert len(scores) == 500 and scores["company_id"].nunique() == 500
assert set(required_output_columns).issubset(scores.columns)

preview_columns = [
    "primary_ticker", "company_name", "sector",
    "structured_score_after_regulatory_penalty", "net_zero_transition_score",
    "overall_coverage_grade",
]
display(scores[preview_columns].head())
display(scores["overall_coverage_grade"].value_counts().rename("companies").to_frame())

,primary_ticker,company_name,sector,structured_score_after_regulatory_penalty,net_zero_transition_score,overall_coverage_grade
0,A,AGILENT TECHNOLOGIES INC,Health Care,55.806307,46.822654,High
1,AAPL,APPLE,Information Technology,41.292020,42.081579,High
2,ABBV,ABBVIE,Health Care,41.091216,70.818408,High
3,ABNB,AIRBNB CLASS A,Consumer Discretionary,64.919197,36.774486,High
4,ABT,ABBOTT LABORATORIES,Health Care,20.925341,49.970069,High


,companies
overall_coverage_grade,
High,424
Low,46
Medium,30


## 5. Test sensitivity and inspect results

Run the same scoring function under alternative assumptions and compare the resulting ranks.

In [39]:
def scenario_block(frame, feature_weights, materiality, threshold, prior_mode):
    """Calculate one pillar block under a selected prior."""
    first = calculate_block(frame, feature_weights, materiality, threshold=threshold, prior=NEUTRAL_PRIOR)
    if prior_mode == "sector_median":
        prior = first["raw"].groupby(frame["sector"]).transform("median").fillna(NEUTRAL_PRIOR)
        return calculate_block(frame, feature_weights, materiality, threshold=threshold, prior=prior)
    return first

def score_scenario(name, env_lambda=ENVIRONMENT_SECTOR_MIX_LAMBDA, threshold=SHRINKAGE_EVIDENCE_THRESHOLD,
                   prior_mode="fixed_50", pillar_weights=None, feature_weights=None, materiality_mode="baseline"):
    """Recalculate final scores under one alternative assumption set."""
    frame = inputs.copy()
    frame["scope12_reported_amount_2024_score"] = (
        env_lambda * frame["scope12_reported_amount_2024_sector_score"]
        + (1 - env_lambda) * frame["scope12_reported_amount_2024_global_score"]
    )
    weights = {key: value.copy() for key, value in FINAL_FEATURE_WEIGHTS.items()}
    for pillar, mapping in (feature_weights or {}).items():
        weights[pillar] = mapping.copy()
    if materiality_mode == "flat":
        materiality = {feature: {sector: 1.0 for sector in SECTORS} for feature in FEATURE_MATERIALITY}
    else:
        materiality = FEATURE_MATERIALITY
    pillar_weights = (pillar_weights or FINAL_PILLAR_WEIGHTS).copy()
    results = {}
    for prefix, title in PILLAR_FEATURE_KEYS.items():
        results[prefix] = scenario_block(frame, weights[title], materiality, threshold, prior_mode)
    score_before = sum(pillar_weights[title] * results[prefix]["adjusted"] for prefix, title in PILLAR_FEATURE_KEYS.items())
    score_after = np.maximum(0, score_before - frame["regulatory_evidence_penalty_provisional"])
    rank = pd.Series(score_after, index=frame.index).rank(method="min", ascending=False).astype(int)
    return pd.DataFrame({"company_id": frame["company_id"], "scenario": name, "scenario_score": score_after, "scenario_rank": rank})

In [40]:
# Run each scenario through the same scoring pipeline.
alternative_feature_weights = {
    "Transition": {"scope12_employee_intensity_trend": 0.40, "sbti_status": 0.40, "climate_governance_support": 0.20},
    "Governance": {"board_independence": 0.45, "ceo_separation": 0.15, "board_attendance": 0.15, "women_executives": 0.15, "sustainability_committee": 0.10},
}
scenario_definitions = [
    ("baseline", {}),
    ("pillar_environment_transition", {"pillar_weights": {"Environmental": 0.50, "Transition": 0.20, "Social": 0.15, "Governance": 0.15}}),
    ("pillar_social_governance", {"pillar_weights": {"Environmental": 0.35, "Transition": 0.10, "Social": 0.275, "Governance": 0.275}}),
    ("environment_lambda_0.25", {"env_lambda": 0.25}),
    ("environment_lambda_0.75", {"env_lambda": 0.75}),
    ("evidence_threshold_0.50", {"threshold": 0.50}),
    ("evidence_threshold_0.90", {"threshold": 0.90}),
    ("neutral_sector_median", {"prior_mode": "sector_median"}),
    ("materiality_flat", {"materiality_mode": "flat"}),
    ("alternative_feature_weights", {"feature_weights": alternative_feature_weights}),
]
sensitivity_parts = [score_scenario(name, **kwargs) for name, kwargs in scenario_definitions]
sensitivity = pd.concat(sensitivity_parts, ignore_index=True)
baseline_sensitivity = sensitivity.loc[sensitivity["scenario"].eq("baseline"), ["company_id", "scenario_score", "scenario_rank"]].rename(
    columns={"scenario_score": "baseline_score", "scenario_rank": "baseline_rank"}
)
sensitivity = sensitivity.merge(baseline_sensitivity, on="company_id", how="left", validate="many_to_one")
sensitivity["rank_change_vs_baseline"] = sensitivity["baseline_rank"] - sensitivity["scenario_rank"]

scenario_correlations = {}
for scenario, group in sensitivity.groupby("scenario"):
    scenario_correlations[scenario] = float(group["scenario_score"].corr(group["baseline_score"], method="spearman"))
sensitivity["spearman_with_baseline"] = sensitivity["scenario"].map(scenario_correlations)
sensitivity = sensitivity.merge(companies[["company_id", "primary_ticker", "company_name", "sector"]], on="company_id", how="left", validate="many_to_one")

In [41]:
baseline_check = sensitivity.loc[sensitivity["scenario"].eq("baseline")].sort_values("company_id")["scenario_score"].to_numpy()
actual_check = scores.sort_values("company_id")["structured_score_after_regulatory_penalty"].to_numpy()
assert np.allclose(baseline_check, actual_check, equal_nan=True)

sensitivity_summary = pd.DataFrame([
    {
        "scenario": scenario,
        "spearman_with_baseline": scenario_correlations[scenario],
        "maximum_absolute_rank_change": int(group["rank_change_vs_baseline"].abs().max()),
        "mean_absolute_rank_change": float(group["rank_change_vs_baseline"].abs().mean()),
    }
    for scenario, group in sensitivity.groupby("scenario", sort=False)
])
display(sensitivity_summary.round(3))

,scenario,spearman_with_baseline,maximum_absolute_rank_change,mean_absolute_rank_change
0,baseline,1.000,0,0.000
1,pillar_environment_transition,0.995,49,11.412
2,pillar_social_governance,0.978,99,22.644
3,environment_lambda_0.25,0.985,92,16.160
4,environment_lambda_0.75,0.985,86,16.172
5,evidence_threshold_0.50,0.999,31,4.140
6,evidence_threshold_0.90,0.999,30,4.696
7,neutral_sector_median,0.988,156,7.784
8,materiality_flat,0.997,43,8.324
9,alternative_feature_weights,0.998,29,6.452


### Inspect leading, trailing, and unusual observations

In [42]:
def confidence_warning(row):
    return "LOW CONFIDENCE - diagnostic only" if row["overall_coverage_grade"] == "Low" else ""

def component_explanation(row):
    components = {
        "E": row["environmental_adjusted_score"], "T": row["transition_adjusted_score"],
        "S": row["social_adjusted_score"], "G": row["governance_adjusted_score"],
    }
    high = max(components, key=components.get)
    low = min(components, key=components.get)
    return f"Strongest {high} {components[high]:.1f}; weakest {low} {components[low]:.1f}; penalty {row['regulatory_evidence_penalty_provisional']:.1f}."

# Keep low-coverage companies out of the compact top/bottom view.
judge_eligible = scores.loc[~scores["overall_coverage_grade"].eq("Low")].copy()
top_table = judge_eligible.nlargest(5, "structured_score_after_regulatory_penalty").copy()
bottom_table = judge_eligible.nsmallest(5, "structured_score_after_regulatory_penalty").copy()
top_table["table_position"] = "Top"
bottom_table["table_position"] = "Bottom"
top_bottom = pd.concat([top_table, bottom_table], ignore_index=True)
top_bottom["component_explanation"] = top_bottom.apply(component_explanation, axis=1)
top_bottom["confidence_warning"] = top_bottom.apply(confidence_warning, axis=1)
display(top_bottom[["table_position", "primary_ticker", "company_name", "structured_score_after_regulatory_penalty", "net_zero_transition_score", "overall_coverage_grade", "component_explanation", "confidence_warning"]])

,table_position,primary_ticker,company_name,structured_score_after_regulatory_penalty,net_zero_transition_score,overall_coverage_grade,component_explanation,confidence_warning
0,Top,VRSK,VERISK ANALYTICS,82.338542,83.955487,High,Strongest E 97.4; weakest S 65.1; penalty 0.0.,
1,Top,VICI,VICI PPTYS INC,79.804539,48.185941,High,Strongest E 95.8; weakest T 51.8; penalty 0.0.,
2,Top,BKNG,BOOKING HOLDINGS,79.216568,84.404255,High,Strongest E 97.7; weakest S 51.1; penalty 0.0.,
3,Top,TPR,TAPESTRY,78.684337,72.980448,High,Strongest S 82.9; weakest G 71.4; penalty 0.0.,
4,Top,ADSK,AUTODESK,78.291546,71.769079,High,Strongest E 92.7; weakest S 53.7; penalty 0.0.,
5,Bottom,DAL,DELTA AIR LINES,9.221414,30.573845,High,Strongest G 42.6; weakest E 1.9; penalty 14.4.,
6,Bottom,ADM,ARCHER DANIELS MIDLAND,9.693775,48.502867,High,Strongest G 55.2; weakest E 5.6; penalty 15.0.,
7,Bottom,WM,WASTE MANAGEMENT INC,11.168199,36.041070,High,Strongest G 47.6; weakest E 6.3; penalty 15.0.,
8,Bottom,MU,MICRON TECHNOLOGY,11.508920,50.366709,High,Strongest T 52.8; weakest E 6.6; penalty 13.4.,
9,Bottom,AMZN,AMAZON.COM INC,14.356592,39.626615,High,Strongest G 45.7; weakest E 1.4; penalty 8.5.,


In [43]:
inspection_parts = []
for prefix, title in PILLAR_FEATURE_KEYS.items():
    columns = ["company_id", "primary_ticker", "company_name", "sector", f"{prefix}_raw", f"{prefix}_adjusted", f"{prefix}_observed_weight", f"{prefix}_confidence"]
    high = inputs.nlargest(5, f"{prefix}_adjusted")[columns].copy()
    low = inputs.nsmallest(5, f"{prefix}_adjusted")[columns].copy()
    high["inspection_type"], high["block"] = "highest", title
    low["inspection_type"], low["block"] = "lowest", title
    inspection_parts.extend([high, low])
gap_inspection = inputs.reindex(inputs["credibility_gap"].abs().sort_values(ascending=False).index).head(10)[[
    "company_id", "primary_ticker", "company_name", "sector", "commitment_score", "realized_transition_score", "credibility_gap",
    "commitment_observed_weight", "realized_transition_observed_weight",
]].copy()
gap_inspection["inspection_type"], gap_inspection["block"] = "largest absolute gap", "Credibility gap"
penalty_inspection = largest_penalties.copy()
penalty_inspection["inspection_type"], penalty_inspection["block"] = "largest penalty", "Regulatory penalty"
manual_inspection = pd.concat(inspection_parts + [gap_inspection, penalty_inspection], ignore_index=True, sort=False)
manual_inspection["inspection_note"] = np.where(
    manual_inspection.get("regulatory_evidence_penalty_provisional", pd.Series(index=manual_inspection.index)).notna(),
    "Review the source-specific evidence used for this penalty.",
    "Interpret the extreme together with observed weight; shrinkage limits sparse-evidence extremes.",
)

## 6. Create figures

Save six compact views used to inspect score composition, coverage, transition results, and rankings.

In [44]:
# Figure 1: score decomposition.
eligible_inputs = inputs.loc[~inputs["overall_coverage_grade"].eq("Low")].copy()
target_score = eligible_inputs["structured_score_after_regulatory_penalty"].median()
target_coverage = eligible_inputs["overall_observed_weight"].median()
representative_index = (
    (eligible_inputs["structured_score_after_regulatory_penalty"] - target_score).abs()
    + 10 * (eligible_inputs["overall_observed_weight"] - target_coverage).abs()
).idxmin()
representative = inputs.loc[representative_index]
decomp_labels = ["Environmental", "Transition", "Social", "Governance"]
decomp_values = [representative[f"{p.lower()}_adjusted"] for p in decomp_labels]
decomp_weights = [FINAL_PILLAR_WEIGHTS[p] for p in decomp_labels]
decomp_contrib = [v * w for v, w in zip(decomp_values, decomp_weights)]
fig, ax = plt.subplots(figsize=(9, 5.2))
bars = ax.barh(decomp_labels, decomp_contrib, color=["#2A9D8F", "#457B9D", "#E9C46A", "#6D597A"])
for bar, score_value, weight in zip(bars, decomp_values, decomp_weights):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f"{score_value:.1f} x {weight:.0%}", va="center", fontsize=10)
ax.set_xlabel("Weighted contribution to structured score")
ax.set_title(f"Score decomposition: {representative['company_name']} ({representative['primary_ticker']})")
ax.text(0.01, -0.20, f"Before penalty {representative['structured_score_before_regulatory_penalty']:.1f}; provisional penalty {representative['regulatory_evidence_penalty_provisional']:.1f}; after penalty {representative['structured_score_after_regulatory_penalty']:.1f}; coverage {representative['overall_observed_weight']:.0%} ({representative['overall_coverage_grade']}).", transform=ax.transAxes, fontsize=9)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_score_decomposition_representative_company.png", dpi=180, bbox_inches="tight")
plt.close(fig)

In [45]:
# Figure 2: coverage by sector and component.
coverage_component_columns = {
    "E: Scope 1+2 amount": "scope12_reported_amount_2024_score",
    "T: Intensity trend": "scope12_employee_intensity_trend_score",
    "T: SBTi": "sbti_status_score",
    "T: Climate oversight": "climate_governance_support_score",
    "S: Diversity": "diversity_score",
    "S: Safety": "employee_safety_score",
    "S: Policies": "social_policy_score",
    "S: Stability": "employee_stability_score",
    "G: Independence": "board_independence_score",
    "G: CEO separation": "ceo_separation_score",
    "G: Attendance": "board_attendance_score",
    "G: Women executives": "women_executives_score",
    "G: Sustainability committee": "sustainability_committee_score",
    "Reg: TRI": "tri_regulatory_adverse_score",
    "Reg: CFPB": "cfpb_regulatory_adverse_score",
    "Reg: CPSC": "cpsc_regulatory_adverse_score",
    "Reg: openFDA": "openfda_regulatory_adverse_score",
}
coverage_heatmap = pd.DataFrame(index=SECTORS)
for label, column in coverage_component_columns.items():
    coverage_heatmap[label] = inputs.assign(observed=inputs[column].notna()).groupby("sector")["observed"].mean().reindex(SECTORS)
fig, ax = plt.subplots(figsize=(17, 7))
sns.heatmap(coverage_heatmap * 100, annot=True, fmt=".0f", cmap="YlGnBu", vmin=0, vmax=100, cbar_kws={"label": "Companies with observed component (%)"}, ax=ax)
ax.set_title("Coverage by sector and component")
ax.set_xlabel("")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_coverage_heatmap_by_sector_component.png", dpi=180, bbox_inches="tight")
plt.close(fig)

In [46]:
# Figure 3: sustainability score versus transition score.
fig, ax = plt.subplots(figsize=(9, 7))
palette = {"High": "#2A9D8F", "Medium": "#E9C46A", "Low": "#E76F51"}
for grade, group in inputs.groupby("overall_coverage_grade"):
    ax.scatter(group["structured_score_after_regulatory_penalty"], group["net_zero_transition_score"], s=38, alpha=0.75, label=f"{grade} coverage", color=palette[grade])
gap_frame = inputs.assign(signed_gap=inputs["structured_score_after_regulatory_penalty"] - inputs["net_zero_transition_score"])
highlight = pd.concat([gap_frame.nlargest(2, "signed_gap"), gap_frame.nsmallest(2, "signed_gap")]).drop_duplicates("company_id")
annotation_offsets = [(12, 24), (12, -24), (-12, 24), (-12, -24)]
for row, (dx, dy) in zip(highlight.itertuples(index=False), annotation_offsets):
    ax.annotate(
        row.primary_ticker,
        (row.structured_score_after_regulatory_penalty, row.net_zero_transition_score),
        xytext=(dx, dy), textcoords="offset points", fontsize=8,
        ha="left" if dx > 0 else "right",
        bbox={"boxstyle": "round,pad=0.15", "fc": "white", "ec": "none", "alpha": 0.8},
        arrowprops={"arrowstyle": "-", "color": "#777777", "lw": 0.6},
    )
ax.set_xlabel("Structured sustainability score after provisional penalty")
ax.set_ylabel("Net-zero transition score")
ax.set_title("Broad sustainability and net-zero transition are distinct")
ax.legend(frameon=True)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_sustainability_vs_net_zero_transition.png", dpi=180, bbox_inches="tight")
plt.close(fig)

In [47]:
# Figure 4: largest regulatory rank changes.
rank_shift_plot = largest_rank_changes.sort_values("rank_delta")
fig, ax = plt.subplots(figsize=(10, 7))
colors = np.where(rank_shift_plot["rank_delta"] < 0, "#D1495B", "#2A9D8F")
ax.barh(rank_shift_plot["primary_ticker"], rank_shift_plot["rank_delta"], color=colors)
ax.axvline(0, color="#333333", linewidth=0.8)
ax.set_xlabel("Rank delta (negative = deterioration after regulatory evidence)")
ax.set_ylabel("")
ax.set_title("Largest rank changes from provisional regulatory evidence")
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_regulatory_rank_shift.png", dpi=180, bbox_inches="tight")
plt.close(fig)

In [48]:
# Figure 5: commitments versus realized outcomes.
fig, ax = plt.subplots(figsize=(9, 7))
points = ax.scatter(inputs["realized_transition_score"], inputs["commitment_score"], c=inputs["credibility_gap"], cmap="RdYlGn_r", vmin=-40, vmax=40, s=42, alpha=0.8)
ax.plot([0, 100], [0, 100], linestyle="--", color="#555555", linewidth=1)
large_gaps = inputs.nlargest(8, "credibility_gap")
ax.scatter(large_gaps["realized_transition_score"], large_gaps["commitment_score"], s=95, facecolors="none", edgecolors="#222222", linewidths=1.0)
label_gaps = large_gaps.head(3)
for row, (dx, dy) in zip(label_gaps.itertuples(index=False), [(28, 32), (42, 0), (28, -32)]):
    ax.annotate(
        row.primary_ticker, (row.realized_transition_score, row.commitment_score),
        xytext=(dx, dy), textcoords="offset points", fontsize=8,
        bbox={"boxstyle": "round,pad=0.15", "fc": "white", "ec": "none", "alpha": 0.85},
        arrowprops={"arrowstyle": "-", "color": "#777777", "lw": 0.6},
    )
ax.set_xlim(0, 100); ax.set_ylim(0, 100)
ax.set_xlabel("Realized transition score")
ax.set_ylabel("Commitment score")
ax.set_title("Commitments versus realized outcomes")
fig.colorbar(points, ax=ax, label="Credibility gap (commitment - realized)")
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_commitments_vs_realized_outcomes.png", dpi=180, bbox_inches="tight")
plt.close(fig)

In [49]:
# Figure 6: top and bottom scores with adequate coverage.
table_view = top_bottom[["table_position", "primary_ticker", "company_name", "structured_score_after_regulatory_penalty", "net_zero_transition_score", "overall_coverage_grade"]].copy()
table_view["structured_score_after_regulatory_penalty"] = table_view["structured_score_after_regulatory_penalty"].map(lambda x: f"{x:.1f}")
table_view["net_zero_transition_score"] = table_view["net_zero_transition_score"].map(lambda x: f"{x:.1f}")
table_view.columns = ["Group", "Ticker", "Company", "Structured", "Net-zero", "Coverage"]
fig, ax = plt.subplots(figsize=(13, 5.4))
ax.axis("off")
table = ax.table(cellText=table_view.values, colLabels=table_view.columns, cellLoc="left", colLoc="center", loc="center", colWidths=[0.08, 0.08, 0.36, 0.11, 0.11, 0.10])
table.auto_set_font_size(False); table.set_fontsize(9); table.scale(1, 1.5)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor("#264653"); cell.set_text_props(color="white", weight="bold")
    elif table_view.iloc[r-1, 0] == "Top":
        cell.set_facecolor("#E8F4F1")
    else:
        cell.set_facecolor("#FCECEE")
ax.set_title("Top and bottom structured scores among Medium/High coverage companies\nComponent explanations are saved in the accompanying CSV", pad=18)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_top_bottom_structured_scores.png", dpi=180, bbox_inches="tight")
plt.close(fig)

print("Representative company:", representative["primary_ticker"], representative["company_name"])
print("Figures created:", len(list(OUTPUT_DIR.glob("*.png"))))

Representative company: SOLV SOLVENTUM
Figures created: 6


## 7. Export and validate

Write the score tables, supporting diagnostics, and final validation record.

In [50]:
# Record the configured weights used by each score.
weight_rows = []
for block, weights in ORIGINAL_FEATURE_WEIGHTS.items():
    for feature, weight in weights.items():
        weight_rows.append({"block": block, "weight_set": "original", "feature": feature, "weight": weight})
for block, weights in FINAL_FEATURE_WEIGHTS.items():
    for feature, weight in weights.items():
        weight_rows.append({"block": block, "weight_set": "final", "feature": feature, "weight": weight})
for feature, weight in ORIGINAL_NET_ZERO_WEIGHTS.items():
    weight_rows.append({"block": "Net-zero transition", "weight_set": "original", "feature": feature, "weight": weight})
for feature, weight in FINAL_NET_ZERO_WEIGHTS.items():
    weight_rows.append({"block": "Net-zero transition", "weight_set": "final", "feature": feature, "weight": weight})
for feature, weight in FINAL_PILLAR_WEIGHTS.items():
    weight_rows.append({"block": "Structured pillar blend", "weight_set": "final", "feature": feature, "weight": weight})
weight_audit = pd.DataFrame(weight_rows)

In [51]:
materiality_rows = []
for feature, mapping in FEATURE_MATERIALITY_LEVELS.items():
    for sector, level in mapping.items():
        materiality_rows.append({"matrix": "feature", "feature_or_source": feature, "sector": sector, "level": level, "multiplier": MATERIALITY_VALUES[level]})
for source, mapping in REGULATORY_MATERIALITY_LEVELS.items():
    for sector, level in mapping.items():
        materiality_rows.append({"matrix": "regulatory", "feature_or_source": source, "sector": sector, "level": level, "multiplier": MATERIALITY_VALUES[level]})
materiality_table = pd.DataFrame(materiality_rows)

In [52]:
coverage_heatmap.reset_index(names="sector").to_csv(OUTPUT_DIR / "structured_score_coverage_by_sector_component.csv", index=False)
feature_coverage_by_sector.to_csv(OUTPUT_DIR / "structured_score_feature_coverage_by_sector.csv", index=False)
correlations.to_csv(OUTPUT_DIR / "structured_score_feature_spearman_correlations.csv")
peer_diagnostics.to_csv(OUTPUT_DIR / "structured_score_peer_fallbacks.csv", index=False)
manual_inspection.to_csv(OUTPUT_DIR / "structured_score_manual_inspection.csv", index=False)
weight_audit.to_csv(OUTPUT_DIR / "structured_score_weight_audit.csv", index=False)
materiality_table.to_csv(OUTPUT_DIR / "structured_score_materiality_matrix.csv", index=False)
top_bottom.to_csv(OUTPUT_DIR / "structured_score_top_bottom_explanations.csv", index=False)
sensitivity.to_csv(OUTPUT_DIR / "structured_score_sensitivity.csv", index=False)

In [53]:
forbidden_ivv_columns = [c for c in inputs.columns if c in {"ivv_market_value_usd", "ivv_weight_pct"}]
assert not forbidden_ivv_columns
inputs.sort_values("company_id").to_csv(OUTPUT_DIR / "structured_score_inputs_500.csv", index=False)
scores.to_csv(OUTPUT_DIR / "structured_scores_500.csv", index=False)
feature_dictionary.to_csv(OUTPUT_DIR / "structured_score_data_dictionary.csv", index=False)

### Run final checks

In [54]:
score_columns = [
    "environmental_raw_score", "environmental_adjusted_score",
    "transition_raw_score", "transition_adjusted_score",
    "social_raw_score", "social_adjusted_score",
    "governance_raw_score", "governance_adjusted_score",
    "structured_score_before_regulatory_penalty", "structured_score_after_regulatory_penalty",
    "net_zero_transition_score", "commitment_score", "realized_transition_score",
]
score_range_pass = all(scores[column].dropna().between(0, 100).all() for column in score_columns)

configured_weight_sums = {
    "final_pillars": sum(FINAL_PILLAR_WEIGHTS.values()),
    **{f"final_{block.lower()}": sum(weights.values()) for block, weights in FINAL_FEATURE_WEIGHTS.items()},
    "final_net_zero": sum(FINAL_NET_ZERO_WEIGHTS.values()),
    "commitment": sum(COMMITMENT_WEIGHTS.values()),
    "realized_transition": sum(REALIZED_TRANSITION_WEIGHTS.values()),
    "regulatory_sources": sum(REGULATORY_SOURCE_WEIGHTS.values()),
}
configured_weights_pass = all(np.isclose(value, 1.0) for value in configured_weight_sums.values())

coverage_columns = [f"{prefix}_observed_weight" for prefix in PILLAR_FEATURE_KEYS] + ["overall_observed_weight"]
observed_weights_pass = all(inputs[column].between(0, 1).all() for column in coverage_columns)

# Test the cleaning rule directly instead of relying on a source-specific zero count.
missing_zero_pass = clean_bloomberg_value(0) == 0 and pd.isna(clean_bloomberg_value("#N/A N/A"))
current_time_basis_pass = (
    scores["general_features_time_basis"].eq(GENERAL_FEATURES_TIME_BASIS).all()
    and scores["quantitative_anchor_year"].eq(2024).all()
)
separate_concepts_pass = not np.allclose(
    scores["structured_score_after_regulatory_penalty"], scores["net_zero_transition_score"]
)

In [55]:
# Final checks cover the calculations that would materially affect the ranking.
required_validation = {
    "exactly_500_unique_company_rows": len(scores) == 500 and scores["company_id"].nunique() == 500,
    "all_503_bloomberg_securities_mapped": len(general_mapped) == 503 and general_mapped["company_id"].notna().all(),
    "share_class_values_agree": not share_class_conflicts,
    "panel_has_6500_unique_company_year_rows": len(panel) == 6500 and not panel.duplicated(["company_id", "observation_year"]).any(),
    "panel_covers_2012_2024": sorted(panel["observation_year"].unique()) == list(range(2012, 2025)),
    "missing_and_zero_are_distinct": bool(missing_zero_pass),
    "ivv_market_value_is_unused": not forbidden_ivv_columns,
    "time_labels_are_correct": bool(current_time_basis_pass),
    "scores_are_within_0_100": bool(score_range_pass),
    "regulatory_penalty_is_within_0_15": bool(scores["regulatory_evidence_penalty_provisional"].between(0, 15).all()),
    "configured_weights_sum_to_one": bool(configured_weights_pass),
    "observed_weights_are_within_0_1": bool(observed_weights_pass),
    "sustainability_and_net_zero_scores_are_separate": bool(separate_concepts_pass),
}
if not all(required_validation.values()):
    raise AssertionError({name: passed for name, passed in required_validation.items() if not passed})

In [56]:
validation.update({
    "feature_summary": {
        "disabled_features": feature_dictionary.loc[
            feature_dictionary["feature_status"].str.startswith("Disabled"),
            ["output_feature", "decision_reason"],
        ].to_dict(orient="records"),
        "redundant_pairs": redundant_pairs,
        "peer_fallback_summary": peer_diagnostics.to_dict(orient="records"),
    },
    "score_summary": {
        "pillar_summary": pillar_summary.to_dict(orient="records"),
        "configured_weight_sums": configured_weight_sums,
        "coverage_grades": scores["overall_coverage_grade"].value_counts().to_dict(),
    },
    "regulatory_summary": {
        "penalty_min": float(inputs["regulatory_evidence_penalty_provisional"].min()),
        "penalty_max": float(inputs["regulatory_evidence_penalty_provisional"].max()),
        "companies_with_observed_evidence": int(inputs["regulatory_penalty_observed_weight"].gt(0).sum()),
        "largest_rank_changes": largest_rank_changes.head(5).to_dict(orient="records"),
    },
    "sensitivity_summary": sensitivity_summary.to_dict(orient="records"),
    "required_validation": required_validation,
})

In [57]:
def json_safe(value):
    """Convert pandas and NumPy values into JSON-safe Python values."""
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if value is pd.NA:
        return None
    return value

with open(OUTPUT_DIR / "structured_score_validation.json", "w", encoding="utf-8") as stream:
    json.dump(json_safe(validation), stream, indent=2, default=str, allow_nan=False)

validation_display = pd.DataFrame({"test": list(required_validation), "passed": list(required_validation.values())})
display(validation_display)
print("All required validations passed.")

,test,passed
0,exactly_500_unique_company_rows,True
1,all_503_bloomberg_securities_mapped,True
2,share_class_values_agree,True
3,panel_has_6500_unique_company_year_rows,True
4,panel_covers_2012_2024,True
5,missing_and_zero_are_distinct,True
6,ivv_market_value_is_unused,True
7,time_labels_are_correct,True
8,scores_are_within_0_100,True
9,regulatory_penalty_is_within_0_15,True


All required validations passed.


In [58]:
required_files = [
    "structured_score_inputs_500.csv",
    "structured_scores_500.csv",
    "structured_score_data_dictionary.csv",
    "structured_score_validation.json",
    "structured_score_sensitivity.csv",
    "01_score_decomposition_representative_company.png",
    "02_coverage_heatmap_by_sector_component.png",
    "03_sustainability_vs_net_zero_transition.png",
    "04_regulatory_rank_shift.png",
    "05_commitments_vs_realized_outcomes.png",
    "06_top_bottom_structured_scores.png",
]

artifact_check = pd.DataFrame([
    {
        "file": filename,
        "exists": (OUTPUT_DIR / filename).exists(),
        "bytes": (OUTPUT_DIR / filename).stat().st_size if (OUTPUT_DIR / filename).exists() else 0,
    }
    for filename in required_files
])

assert artifact_check["exists"].all() and artifact_check["bytes"].gt(0).all()
print(f"Complete: {len(scores)} company scores and {len(sensitivity):,} sensitivity rows.")

Complete: 500 company scores and 5,000 sensitivity rows.
